# ISLES'24 one-click 16-worker persistent downloader

This notebook is intentionally minimal and does **not** clone the repository.

Run **Runtime → Run all**. It mounts Google Drive, fetches the latest standalone downloader script, inspects the existing persistent prefix/chunks, and continues the download with **16 parallel workers**.

Existing `train.7z.prefix`, completed chunks, and partial chunks are preserved and reused.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import subprocess

ARCHIVE = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024/raw/train.7z')
PARTS = Path(str(ARCHIVE) + '.parts')
PREFIX = Path(str(ARCHIVE) + '.prefix')
SCRIPT = Path('/content/download_isles2024_parallel.py')
ARCHIVE.parent.mkdir(parents=True, exist_ok=True)

def gib(n):
    return n / (1024**3)

print('=== EXISTING PERSISTENT STATE ===')
saved = 0
if PREFIX.exists():
    prefix_size = PREFIX.stat().st_size
    saved += prefix_size
    print(f'Prefix: {gib(prefix_size):.2f} GiB')
else:
    print('Prefix: none')

if PARTS.exists():
    chunk_files = [p for p in PARTS.iterdir() if p.is_file() and p.name != 'manifest.json']
    chunk_bytes = sum(p.stat().st_size for p in chunk_files)
    saved += chunk_bytes
    complete_names = sum(p.name.endswith('.bin') for p in chunk_files)
    partial_names = sum(p.name.endswith('.partial') for p in chunk_files)
    print(f'Chunks: {complete_names} .bin files, {partial_names} .partial files, {gib(chunk_bytes):.2f} GiB stored')
else:
    print('Chunks: none yet')
print(f'Total bytes currently visible in Drive: ~{gib(saved):.2f} GiB')

print('\n=== FETCH LATEST DOWNLOADER SCRIPT ===')
RAW_URL = 'https://raw.githubusercontent.com/NTinkicht/HybridStrokeSeg-Reproduction/main/scripts/download_isles2024_parallel.py'
subprocess.run([
    'curl','-fL','--retry','20','--retry-delay','3','--retry-all-errors',
    RAW_URL,'-o',str(SCRIPT)
], check=True)
print('Downloader script ready:', SCRIPT)

print('\n=== START / RESUME DOWNLOAD WITH 16 WORKERS ===')
subprocess.run([
    'python', str(SCRIPT),
    '--archive', str(ARCHIVE),
    '--parts-dir', str(PARTS),
    '--workers', '16',
    '--part-size-gib', '1',
    '--retries', '-1',
], check=True)

print('\n=== FINAL STATE ===')
print('Archive exists:', ARCHIVE.exists())
if ARCHIVE.exists():
    print(f'Archive size: {gib(ARCHIVE.stat().st_size):.2f} GiB')
print('Parts directory exists:', PARTS.exists())
print('Successful completion means the final archive passed the official MD5 check.')


## If Colab disconnects

Open this same notebook and choose **Runtime → Run all** again. Do not delete the `.prefix`, `.parts`, `.bin`, or `.partial` files.
